<a href="https://colab.research.google.com/github/Ninja12309/Section1_SQL_in_R/blob/main/NorthStar_Section1_SQL_in_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install R environment in Colab

In [7]:
# install R kernel support
!pip install rpy2

# Load R environment

In [2]:
%load_ext rpy2.ipython

In [3]:
from google.colab import files

uploaded = files.upload()

Saving app_events.csv to app_events.csv
Saving complaints.csv to complaints.csv
Saving customers.csv to customers.csv
Saving data_dictionary.csv to data_dictionary.csv
Saving deliveries.csv to deliveries.csv
Saving drivers.csv to drivers.csv
Saving hubs.csv to hubs.csv
Saving incidents.csv to incidents.csv
Saving orders.csv to orders.csv
Saving vehicles.csv to vehicles.csv


# Import libraries and datasets in R

In [11]:
%%R

# install required libraries if not already installed

install.packages("sqldf")
install.packages("dplyr")

library(sqldf)
library(dplyr)

# load datasets


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/sqldf_0.4-12.tar.gz'
Content type 'application/x-gzip' length 61077 bytes (59 KB)
downloaded 59 KB


The downloaded source packages are in
	‘/tmp/Rtmpp9OUKW/downloaded_packages’
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/dplyr_1.2.0.tar.gz'
Content type 'application/x-gzip' length 922393 bytes (900 KB)
downloaded 900 KB


The downloaded source packages are in
	‘/tmp/Rtmpp9OUKW/downloaded_packages’


In [17]:
%%R

customers <- read.csv("customers.csv")
orders <- read.csv("orders.csv")
drivers <- read.csv("drivers.csv")
vehicles <- read.csv("vehicles.csv")
complaints <- read.csv("complaints.csv")
app_events <- read.csv("app_events.csv")
incidents <- read.csv("incidents.csv")
hubs <- read.csv("hubs.csv")

# preview data

head(customers)
head(orders)

  order_id customer_id service_type    order_created_at promised_window_hours
1   O00001       C0292    Passenger 2024-08-20 14:43:00                     6
2   O00002       C0459    Passenger 2024-05-14 22:16:00                    24
3   O00003       C0161    Passenger 2025-09-02 14:37:00                     4
4   O00004       C0520       Parcel 2025-01-11 17:15:00                     2
5   O00005       C0558       Retail 2025-02-17 19:32:00                    12
6   O00006       C0437       Retail 2024-08-05 04:55:00                     1
  pickup_zone dropoff_zone priority_level order_value booking_channel
1     Airport        South         Medium      126.65             App
2       North      AIRPORT            Low      109.30             App
3        West      AIRPORT           High       33.50           Phone
4   RiverSide        North         Medium       10.04             App
5   Riverside        SOUTH            Low      125.58           Phone
6     CENTRAL         East        

# SQL Query 1

Analyse demand distribution by service type

Business problem:
NorthStar management needs to understand which services generate the highest demand in order to allocate resources effectively.

In [20]:
%%R

query1 <- sqldf('

SELECT
    service_type,
    COUNT(order_id) AS total_orders,
    ROUND(AVG(order_value),2) AS avg_order_value

FROM orders

GROUP BY service_type

ORDER BY total_orders DESC

')

query1

  service_type total_orders avg_order_value
1    Passenger          341           96.07
2       Parcel          308           87.62
3       Retail          297           90.01
4     Business          165           92.25
5      Medical          139           87.14


# SQL Query 2

Identify main causes of customer dissatisfaction

Business problem:
Understanding the most common complaint types helps identify operational weaknesses.

In [21]:
%%R

query2 <- sqldf('

SELECT
    complaint_type,
    COUNT(complaint_id) AS total_complaints,
    ROUND(AVG(compensation_amount),2) AS avg_compensation

FROM complaints

GROUP BY complaint_type

ORDER BY total_complaints DESC

')

query2

     complaint_type total_complaints avg_compensation
1             Delay              101            18.05
2      MissedPickup               64            22.59
3          AppIssue               53            19.61
4   DriverBehaviour               51            21.15
5 SupportExperience               20            17.13
6           Billing               16            23.87
7            Damage               15            23.98


# SQL Query 3

Analyse performance by geographic zone

Business problem:
Certain zones may experience more operational inefficiencies than others.

In [22]:
%%R

query3 <- sqldf('

SELECT
    pickup_zone,
    COUNT(order_id) AS total_orders,
    ROUND(AVG(order_value),2) AS avg_order_value

FROM orders

GROUP BY pickup_zone

ORDER BY total_orders DESC

')

query3

   pickup_zone total_orders avg_order_value
1         East          104           92.22
2        South          103           92.40
3         EAST          103           91.33
4    RiverSide           86           80.38
5      Airport           85          108.85
6         WEST           84           89.20
7          Ctr           80           94.50
8      Central           79           77.20
9      CENTRAL           79           93.58
10       SOUTH           78           88.18
11        West           71           87.18
12   Riverside           65           91.90
13       north           64           96.46
14       NORTH           60           89.35
15     AIRPORT           59           96.74
16       North           50           86.10


# SQL Query 4

Evaluate driver workforce experience levels

Business problem:
Driver capability may influence delivery performance and service quality.

In [23]:
%%R

query4 <- sqldf('

SELECT
    base_zone,
    COUNT(driver_id) AS total_drivers,
    ROUND(AVG(years_experience),2) AS avg_experience,
    ROUND(AVG(driver_rating),2) AS avg_rating

FROM drivers

GROUP BY base_zone

ORDER BY avg_rating DESC

')

query4

   base_zone total_drivers avg_experience avg_rating
1  RiverSide            10           6.80       4.33
2      South            21           8.14       4.28
3    CENTRAL            10           8.10       4.26
4      north            11           7.91       4.25
5        Ctr             6           5.00       4.25
6    Airport             9           9.00       4.25
7       East            14           9.00       4.22
8      SOUTH             8           7.88       4.20
9    AIRPORT            10           9.50       4.18
10      West            10           8.90       4.14
11     NORTH            12           6.50       4.14
12   Central            12           9.83       4.14
13      WEST            10           9.40       4.09
14 Riverside             7           8.29       4.09
15     North            13           8.69       3.90
16      EAST             7           6.57       3.90


# SQL Query 5

Identify high priority operational workload

Business problem:
High priority orders may contribute to operational pressure and delays.

In [24]:
%%R

query5 <- sqldf('

SELECT
    priority_level,
    COUNT(order_id) AS total_orders,
    ROUND(AVG(promised_window_hours),2) AS avg_delivery_window

FROM orders

GROUP BY priority_level

ORDER BY total_orders DESC

')

query5

  priority_level total_orders avg_delivery_window
1         Medium          503                7.71
2            Low          348                7.68
3           High          308                7.73
4       Critical           91                6.54


# SQL Query 6

Link complaints to service types

Business problem:
Identifying which services generate the most complaints helps prioritise operational improvements.

In [25]:
%%R

query6 <- sqldf('

SELECT
    o.service_type,
    COUNT(c.complaint_id) AS total_complaints

FROM orders o

JOIN complaints c

ON o.order_id = c.order_id

GROUP BY o.service_type

ORDER BY total_complaints DESC

')

query6

  service_type total_complaints
1    Passenger               84
2       Retail               83
3       Parcel               77
4     Business               39
5      Medical               37


# Query 7

Customer engagement behaviour

Business problem:
understand which device platforms customers use most

In [26]:
%%R

query7 <- sqldf('

SELECT
    device_type,
    COUNT(event_id) AS total_events,
    ROUND(AVG(api_latency_ms),2) AS avg_latency

FROM app_events

GROUP BY device_type

ORDER BY total_events DESC

')

query7

  device_type total_events avg_latency
1     Android          315      464.90
2         iOS          233      463.15
3         Web           92      474.68
